### Pinecone Vector Database



In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
missing = [
    name
    for name, value in {
        "PINECONE_API_KEY": PINECONE_API_KEY,
        "OPENAI_API_KEY": OPENAI_API_KEY,
    }.items()
    if not value
]
USE_PINECONE = not missing
if missing:
    print(f"Missing {', '.join(missing)}. Using local fallback vector store for this run.")
else:
    print("Pinecone credentials found. Using the real Pinecone path.")

api_key = PINECONE_API_KEY

class HashEmbeddings(Embeddings):
    """Deterministic local fallback embeddings for offline notebook runs."""

    def __init__(self, n_features=1024):
        self.vectorizer = HashingVectorizer(
            n_features=n_features,
            alternate_sign=False,
            norm="l2",
            ngram_range=(1, 2),
        )

    def _embed(self, texts):
        return self.vectorizer.transform(texts).toarray().astype(float).tolist()

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed([text])[0]

def source_filter(source):
    return lambda doc: doc.metadata.get("source") == source

In [ ]:
print("Install cloud dependencies only if you plan to use real Pinecone credentials:")
print("uv add langchain-pinecone pinecone langchain-openai")

In [ ]:
if USE_PINECONE:
    try:
        from pinecone import Pinecone
        pc = Pinecone(api_key=api_key)
    except ImportError:
        USE_PINECONE = False
        pc = None
        print("pinecone is not installed. Falling back to InMemoryVectorStore.")
else:
    pc = None

pc

In [ ]:
if OPENAI_API_KEY:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=1024,
        api_key=OPENAI_API_KEY,
    )
else:
    embeddings = HashEmbeddings(n_features=1024)

embeddings

In [ ]:
index_name = "rag"  # change if desired
index = None

if USE_PINECONE:
    from pinecone import ServerlessSpec

    if not pc.has_index(index_name):
        pc.create_index(
            name=index_name,
            dimension=1024,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    index = pc.Index(index_name)
else:
    print("Using local fallback store; no Pinecone index was created.")

In [ ]:
index

In [ ]:
USING_FALLBACK_VECTOR_STORE = False

if USE_PINECONE and index is not None:
    try:
        from langchain_pinecone import PineconeVectorStore
        vector_store = PineconeVectorStore(index=index, embedding=embeddings)
    except ImportError:
        USE_PINECONE = False
        print("langchain_pinecone is not installed. Falling back to InMemoryVectorStore.")

if not USE_PINECONE:
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embedding=embeddings)
    USING_FALLBACK_VECTOR_STORE = True

vector_store

In [ ]:
vector_store

In [ ]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building a LangChain project with retrieval examples.",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
vector_store.add_documents(documents=documents)

In [ ]:
### Query Directly
tweet_filter = source_filter("tweet") if USING_FALLBACK_VECTOR_STORE else {"source": "tweet"}
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter=tweet_filter,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
news_filter = source_filter("news") if USING_FALLBACK_VECTOR_STORE else {"source": "news"}
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow", k=1, filter=news_filter
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

In [ ]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1, "filter": news_filter},
)
retriever.invoke("Stealing from the bank is a crime")